# Quickstart: HK_DEV_SHARED Fi-NeMo + DeepISA rerun

This Colab constructs the exact sharing rerun cohort, admits model-specific dinucleotide DeepLIFT/SHAP attributions only after sequence-identity validation, then runs resumable Fi-NeMo and DeepISA stages. It does **not** modify frozen Figure 1–6 assets.

Before every heavy stage, inspect the displayed sources, cohort counts and parameters, then explicitly set `CONFIRM = 'RUN'`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Clone once, or set REPO to an existing clone.
REPO = '/content/Epromoter_grammar_decpher'
!test -d $REPO/.git || git clone https://github.com/JoneSu1/Epromoter_grammar_decpher.git $REPO
%cd $REPO
!git pull --ff-only


In [ ]:
# Pinned runtime family for the serialized Keras models and Fi-NeMo.
!pip -q install 'numpy<2' 'tensorflow==2.15.1' pandas h5py loguru bioframe finemo leidenalg igraph numba

import json, os, subprocess, sys
CONFIG = f'{REPO}/configs/hk_dev_shared_rerun_config.json'
PIPELINE = f'{REPO}/scripts/hk_dev_shared_pipeline.py'
DATA_ROOT = '/content/drive/MyDrive/DeepEpromote/Drosophila'
OUT_ROOT = f'{DATA_ROOT}/HK_DEV_SHARED_rerun_202609'

def run(*args):
    command = [sys.executable, PIPELINE, '--config', CONFIG, '--data-root', DATA_ROOT, '--output-root', OUT_ROOT, *args]
    print(' '.join(command))
    subprocess.run(command, check=True)


## Gate 1 — review source data, cohort and fixed parameters

Expected counts: sharing `17,380`; proximal/core `12,260`; distal `5,120`; observed CAGE proximal/core `4,178`. The report also fingerprints every table and reports the five scan tracks. Stop if any count differs.

In [ ]:
run('inspect')
CONFIRM = 'REVIEW'  # Change exactly to RUN only after reviewing the report.
assert CONFIRM in {'REVIEW', 'RUN'}


In [ ]:
if CONFIRM == 'RUN':
    run('prepare')
else:
    print('Preparation held for review. Set CONFIRM = RUN and re-run this cell.')


## Gate 2 — model-specific attribution

Generate the five `sequences` + `hyp_scores` NPZ files using the reviewed upstream dinucleotide DeepLIFT/SHAP procedure (`100` dinucleotide backgrounds; batch `20`). Do not use a generic substitute. Save them below the run directory, then inspect their shape and provenance before admission.

In [ ]:
ATTRIBUTIONS = {
    's3_hk': f'{OUT_ROOT}/reviewed_attributions/s3_hk.npz',
    's3_dev': f'{OUT_ROOT}/reviewed_attributions/s3_dev.npz',
    'deepisa_hk': f'{OUT_ROOT}/reviewed_attributions/deepisa_hk.npz',
    'deepisa_dev': f'{OUT_ROOT}/reviewed_attributions/deepisa_dev.npz',
    'deepisa_cage': f'{OUT_ROOT}/reviewed_attributions/deepisa_cage.npz',
}
for track, path in ATTRIBUTIONS.items():
    print(track, path, 'EXISTS' if os.path.exists(path) else 'MISSING')

# Keep REVIEW until all five files exist and their provenance has been checked.
CONFIRM = 'REVIEW'


In [ ]:
if CONFIRM == 'RUN':
    for track, path in ATTRIBUTIONS.items():
        run('admit-attributions', '--track', track, '--source', path)
else:
    print('Attribution admission held for review.')


## Gate 3 — Fi-NeMo scanning

Confirm that S3 tracks use their separate HK/DEV motif H5 files across all `17,380` windows, while the three DeepISA tracks use the shared 24-bp atlas and proximal/core inputs only. The pipeline validates 249 bp before trimming one final base to 248 bp.

In [ ]:
CONFIRM = 'REVIEW'
if CONFIRM == 'RUN':
    for track in ATTRIBUTIONS:
        run('scan', '--track', track)
else:
    print('Fi-NeMo scans held for review.')


## Gate 4 — DeepISA

Run only the three proximal/core shared-atlas tracks. Set `EP_ISA_SOURCE` to the directory containing the reviewed `Ep_ISA_NEW/` implementation. Restart an interrupted run at the earliest missing stage with `--start-from`; existing valid checkpoints are retained.

In [ ]:
EP_ISA_SOURCE = f'{DATA_ROOT}/DeepISA/Ep_ISA_NEW_src'
CONFIRM = 'REVIEW'
if CONFIRM == 'RUN':
    for track in ('deepisa_hk', 'deepisa_dev', 'deepisa_cage'):
        run('deepisa', '--track', track, '--isa-source', EP_ISA_SOURCE, '--start-from', 'preflight_audit')
else:
    print('DeepISA held for review.')


Successful stages are recorded in `OUT_ROOT/state/`. Re-running unchanged stages resumes safely. See `docs/HK_DEV_SHARED_COLAB_QUICKSTART.md` for the contracts, expected artifacts and restart semantics.